In [1]:
import json
import re
import sqlite3

In [2]:
topic = "Linux"

In [3]:
def slugify_title(title):
    return re.sub(r"[^a-z0-9]+", "_", title.lower()).strip("_")


def get_connection(db_path="../quiz_outlines.db"):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


In [4]:
def get_outline(conn, article_title):
    row = conn.execute(
        "SELECT article_id, outline_json FROM outlines WHERE article_title = ?",
        (article_title,),
    ).fetchone()
    if row is None:
        return None
    return {
        "article_id": row["article_id"],
        "outline": json.loads(row["outline_json"])
    }


In [5]:
conn = get_connection()
outline = get_outline(conn, topic)

In [6]:
outline

{'article_id': 'linux',
 'outline': {'article_title': 'Linux',
  'sections': [{'breadcrumb': 'Introduction',
    'preview': 'Linux ( LIN-uuks) is a family of free and open-source software Unix-like operating systems based on the Linux kernel, which was first released on 17...',
    'token_count': 412},
   {'breadcrumb': 'Overview',
    'preview': 'The Linux kernel was created by Linus Torvalds, following the lack of a working kernel for GNU, a Unix-compatible operating system made entirely of...',
    'token_count': 590},
   {'breadcrumb': 'History > Precursors',
    'preview': "The Unix operating system was conceived of and implemented in 1969, at AT&T's Bell Labs in the United States, by Ken Thompson, Dennis Ritchie,...",
    'token_count': 537},
   {'breadcrumb': 'History > Creation',
    'preview': 'While attending the University of Helsinki in the fall of 1990, Torvalds enrolled in a Unix course. The course used a MicroVAX minicomputer running...',
    'token_count': 442},
   {'br

In [ ]:
from pydantic import BaseModel

class QuizOutline(BaseModel):
    breadcrumb: str
    questions: int
    difficulty: str
    reason: str

In [7]:
PLANNER_SYSTEM_PROMPT = """
You are an assessment planner.

Your task is to design a blueprint for a quiz using only the article outline that you are given.

The outline contains:
- hierarchical section names (breadcrumbs)
- a short preview of each section
- the approximate size of each section

Do NOT generate quiz questions.
Do NOT retrieve information.
Do NOT invent facts that are not implied by the outline.

Your job is only to decide:

1. Which sections should contribute questions.
2. How many questions should come from each section.
3. What difficulty each section should contribute.
4. Why each section was selected.

When creating the blueprint:

- Prefer broad coverage over concentrating questions in a single section.
- Ensure every selected section appears relevant to the user's request.
- Avoid selecting sections that appear too small or too narrow unless they are specifically relevant.
- Large sections may receive multiple questions.
- Introductory sections should usually receive fewer questions than substantive sections.
- If the user requests an easier quiz, favor foundational sections.
- If the user requests a harder quiz, favor advanced or specialized sections.
- The total number of planned questions MUST equal the requested number.

Return ONLY valid JSON.
"""

In [8]:
PLANNER_PROMPT = f"""
User request:

Generate a medium-difficulty quiz about Linux.

Number of questions:
10

Article outline:

{outline}
"""

In [13]:
from google import genai
from google.genai import types
import os

In [15]:
gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [43]:
response = gemini_client.models.generate_content(model="gemini-3.5-flash", config=types.GenerateContentConfig(system_instruction=PLANNER_SYSTEM_PROMPT), contents=PLANNER_PROMPT)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [34]:
blueprint = response.text.strip().replace("```json", "").replace("```", "").strip()
blueprint = json.loads(blueprint).get("blueprint")
blueprint

[{'breadcrumb': 'Introduction',
  'questions': 1,
  'difficulty': 'easy',
  'reason': 'Provides an essential foundational question regarding the definition and baseline timeline of the Linux operating system family.'},
 {'breadcrumb': 'Overview',
  'questions': 1,
  'difficulty': 'medium',
  'reason': 'A substantial section that explores the relationship between the Linux kernel and the GNU project, which is core to understanding Linux operating systems.'},
 {'breadcrumb': 'History > Precursors',
  'questions': 1,
  'difficulty': 'medium',
  'reason': 'Covers the historical roots of Unix at Bell Labs, which is crucial for testing medium-difficulty historical context.'},
 {'breadcrumb': 'History > Copyright, trademark, and naming',
  'questions': 1,
  'difficulty': 'medium',
  'reason': 'A large and important section discussing the GPL v2 license and distribution requirements, key to open-source governance.'},
 {'breadcrumb': 'Usage > Market share and uptake',
  'questions': 1,
  'diffi

In [17]:
blueprint = [
    {
        "breadcrumb": "Introduction",
        "questions": 1,
        "difficulty": "easy",
        "reason": "Provides an essential foundational question regarding the definition and baseline timeline of the Linux operating system family.",
    },
    {
        "breadcrumb": "Overview",
        "questions": 1,
        "difficulty": "medium",
        "reason": "A substantial section that explores the relationship between the Linux kernel and the GNU project, which is core to understanding Linux operating systems.",
    },
    {
        "breadcrumb": "History > Precursors",
        "questions": 1,
        "difficulty": "medium",
        "reason": "Covers the historical roots of Unix at Bell Labs, which is crucial for testing medium-difficulty historical context.",
    },
    {
        "breadcrumb": "History > Copyright, trademark, and naming",
        "questions": 1,
        "difficulty": "medium",
        "reason": "A large and important section discussing the GPL v2 license and distribution requirements, key to open-source governance.",
    },
    {
        "breadcrumb": "Usage > Market share and uptake",
        "questions": 1,
        "difficulty": "medium",
        "reason": "One of the largest sections in the text, detailing how Linux has performed in quantitative studies, market share, and reliability.",
    },
    {
        "breadcrumb": "Design",
        "questions": 2,
        "difficulty": "medium",
        "reason": "The largest overall section, detailing the evolutionary philosophy of the Linux kernel design. Two questions are allocated here to capture different facets of this design philosophy.",
    },
    {
        "breadcrumb": "Design > User interface",
        "questions": 1,
        "difficulty": "medium",
        "reason": "Covers user interface paradigms (CLI, GUI, shells) which are highly relevant to intermediate Linux users.",
    },
    {
        "breadcrumb": "Development > Community",
        "questions": 1,
        "difficulty": "easy",
        "reason": "Examines the distribution models and community/vendor ecosystem, providing a softer transition into development topics.",
    },
    {
        "breadcrumb": "Development > Programming on Linux",
        "questions": 1,
        "difficulty": "hard",
        "reason": "Tests technical details on programming language support, toolchains, and original development tools on Linux.",
    },
]

In [ ]:
blueprint = [{k: v for k, v in bp.items() if k != "reason"} for bp in blueprint]

In [ ]:
from qdrant_client import QdrantClient, models

client = QdrantClient(url="http://localhost:6333")

res = client.scroll(
    collection_name="Quiz-App-Dev-Collection",
    scroll_filter=models.Filter(
        must=[
            models.FieldCondition(
                key="section_breadcrumb",
                match=models.MatchValue(value=blueprint[6]["breadcrumb"]),
            ),
        ]
    ),
)


In [39]:
res

([Record(id=10, payload={'article_title': 'Linux', 'section_title': 'User interface', 'section_breadcrumb': 'Design > User interface', 'source_url': 'https://en.wikipedia.org/wiki/Linux#User_interface', 'raw_text': 'The user interface, also known as the shell, is either a command-line interface (CLI), a graphical user interface (GUI), or controls attached to the associated hardware, which is common for embedded systems. For desktop systems, the default user interface is usually graphical, although the CLI is commonly available through terminal emulator windows or on a separate virtual console.\nCLI shells are text-based user interfaces, which use text for both input and output. The dominant shell used in Linux is the Bourne-Again Shell (bash), originally developed for the GNU Project; other shells such as Zsh are also used. Most low-level Linux components, including various parts of the userland, use the CLI exclusively. The CLI is particularly suited for automation of repetitive or de

In [24]:
blueprint[2]["breadcrumb"]

'History > Precursors'

In [41]:
def batch_blueprint(blueprint, items_per_batch=3):
    return [
        blueprint[i : i + items_per_batch]
        for i in range(0, len(blueprint), items_per_batch)
    ]


In [47]:
blueprint

[{'breadcrumb': 'Introduction', 'questions': 1, 'difficulty': 'easy'},
 {'breadcrumb': 'Overview', 'questions': 1, 'difficulty': 'medium'},
 {'breadcrumb': 'History > Precursors',
  'questions': 1,
  'difficulty': 'medium'},
 {'breadcrumb': 'History > Copyright, trademark, and naming',
  'questions': 1,
  'difficulty': 'medium'},
 {'breadcrumb': 'Usage > Market share and uptake',
  'questions': 1,
  'difficulty': 'medium'},
 {'breadcrumb': 'Design', 'questions': 2, 'difficulty': 'medium'},
 {'breadcrumb': 'Design > User interface',
  'questions': 1,
  'difficulty': 'medium'},
 {'breadcrumb': 'Development > Community',
  'questions': 1,
  'difficulty': 'easy'},
 {'breadcrumb': 'Development > Programming on Linux',
  'questions': 1,
  'difficulty': 'hard'}]

In [60]:
for item in blueprint:
    retrieved_context = client.scroll(
        collection_name="Quiz-App-Dev-Collection",
        scroll_filter=models.Filter(
            must=[
                models.FieldCondition(
                    key="section_breadcrumb",
                    match=models.MatchValue(value=item["breadcrumb"]),
                ),
            ]
        ),
    )
    # print(retrieved_context[0][0].payload)
    item["article_title"] = retrieved_context[0][0].payload["article_title"]
    item["text"] = retrieved_context[0][0].payload["raw_text"]
    item["source_url"] = retrieved_context[0][0].payload["source_url"]



In [61]:
blueprint

[{'breadcrumb': 'Introduction',
  'questions': 1,
  'difficulty': 'easy',
  'article_title': 'Linux',
  'text': 'Linux ( LIN-uuks) is a family of free and open-source software Unix-like operating systems based on the Linux kernel, which was first released on 17 September 1991 by Linus Torvalds. Some members of the family are typically packaged as a distribution (a.k.a. distro), which includes the kernel alongside supporting system software and libraries developed by third parties—such as GNU, Red Hat, and X.Org—to create a complete operating system; however, not all Linux-based operating systems are considered distros, with Android being an example. Linux was originally designed as a clone of Unix and is distributed under the copyleft GPL license.\nThere are many thousands of Linux distributions, many based directly or indirectly on other distributions; popular Linux distros include Debian, Fedora Linux, Linux Mint, Arch Linux, and Ubuntu, while commercial distributions include Red Hat

In [62]:
blueprint = [{k: v for k, v in bp.items() if k != "reason"} for bp in blueprint]
batch_blueprint(blueprint)


[[{'breadcrumb': 'Introduction',
   'questions': 1,
   'difficulty': 'easy',
   'article_title': 'Linux',
   'text': 'Linux ( LIN-uuks) is a family of free and open-source software Unix-like operating systems based on the Linux kernel, which was first released on 17 September 1991 by Linus Torvalds. Some members of the family are typically packaged as a distribution (a.k.a. distro), which includes the kernel alongside supporting system software and libraries developed by third parties—such as GNU, Red Hat, and X.Org—to create a complete operating system; however, not all Linux-based operating systems are considered distros, with Android being an example. Linux was originally designed as a clone of Unix and is distributed under the copyleft GPL license.\nThere are many thousands of Linux distributions, many based directly or indirectly on other distributions; popular Linux distros include Debian, Fedora Linux, Linux Mint, Arch Linux, and Ubuntu, while commercial distributions include Re

In [64]:
blueprint

[{'breadcrumb': 'Introduction',
  'questions': 1,
  'difficulty': 'easy',
  'article_title': 'Linux',
  'text': 'Linux ( LIN-uuks) is a family of free and open-source software Unix-like operating systems based on the Linux kernel, which was first released on 17 September 1991 by Linus Torvalds. Some members of the family are typically packaged as a distribution (a.k.a. distro), which includes the kernel alongside supporting system software and libraries developed by third parties—such as GNU, Red Hat, and X.Org—to create a complete operating system; however, not all Linux-based operating systems are considered distros, with Android being an example. Linux was originally designed as a clone of Unix and is distributed under the copyleft GPL license.\nThere are many thousands of Linux distributions, many based directly or indirectly on other distributions; popular Linux distros include Debian, Fedora Linux, Linux Mint, Arch Linux, and Ubuntu, while commercial distributions include Red Hat

In [63]:
def build_batch_prompt(sections_with_meta):
    blocks = []
    for i, item in enumerate(sections_with_meta, start=1):
        blocks.append(f"""Section {i}:
Article: {item["article_title"]}
Breadcrumb: {item["breadcrumb"]}
Difficulty: {item["difficulty"]}
Number of questions required: {item["questions"]}

Passage:
\"\"\"
{item["text"]}
\"\"\"""")

    joined = "\n\n---\n\n".join(blocks)
    return f"""You will generate quiz questions for multiple sections of an article in one pass.

{joined}

Rules:
- Each question must be answerable strictly from its OWN section's passage above. Do not blend facts across sections.
- Generate exactly the required number of questions for each section.
- Tag every question with the correct section_index (the number labeled "Section N" above).
- Exactly 4 options per question, one correct.
- Distractors must be plausible but clearly wrong per the passage.
- If a section requires more than one question, each must test a DIFFERENT fact from that passage.
- easy = direct fact recall. medium = connects two details. hard = subtle distinction or inference."""


In [68]:
batched_blueprint = batch_blueprint(blueprint)

In [69]:
len(batched_blueprint)

3

In [73]:
for blueprint in batched_blueprint:
    print(build_batch_prompt(blueprint))
    print("=========")

You will generate quiz questions for multiple sections of an article in one pass.

Section 1:
Article: Linux
Breadcrumb: Introduction
Difficulty: easy
Number of questions required: 1

Passage:
"""
Linux ( LIN-uuks) is a family of free and open-source software Unix-like operating systems based on the Linux kernel, which was first released on 17 September 1991 by Linus Torvalds. Some members of the family are typically packaged as a distribution (a.k.a. distro), which includes the kernel alongside supporting system software and libraries developed by third parties—such as GNU, Red Hat, and X.Org—to create a complete operating system; however, not all Linux-based operating systems are considered distros, with Android being an example. Linux was originally designed as a clone of Unix and is distributed under the copyleft GPL license.
There are many thousands of Linux distributions, many based directly or indirectly on other distributions; popular Linux distros include Debian, Fedora Linux,

In [74]:
questions = ""
for blueprint in batched_blueprint:
    response = gemini_client.models.generate_content(
        model="gemini-3.5-flash",
        # config=types.GenerateContentConfig(system_instruction=PLANNER_SYSTEM_PROMPT),
        contents=build_batch_prompt(blueprint),
    )
    questions += response.text


In [75]:
with open("questions.txt", "w") as f:
    f.write(questions)